# News label quality check (news-sentiment track)

Compares the LLM labels from `cryptosense.news.label` with hand labels on a random sample of 25 articles, 5 per source, seed 0.

- **Stage 1:** draw the sample and write `data/news/qc_sample.csv` with empty `my_*` columns for the hand labels. No LLM calls; no model labels are shown.
- **Stage 2:** label the sample twice with the same model and prompt, writing only to `data/news/qc_labels.csv`, and compare.

## Step 0: Imports and settings

In [ ]:
from pathlib import Path

import pandas as pd

from cryptosense.news.qc import add_missing_label_columns, draw_qc_sample, write_qc_sample

DATA_DIR = Path.cwd().parent / "data"
NEWS_DIR = DATA_DIR / "news"
PER_SOURCE = 5
SEED = 0

## Step 1: Draw the sample and write it for hand labeling

The sample is drawn only if `qc_sample.csv` does not exist yet; `write_qc_sample` also refuses to overwrite it, so hand labels are never replaced by re-running this cell.

In [ ]:
QC_SAMPLE_PATH = NEWS_DIR / "qc_sample.csv"
if QC_SAMPLE_PATH.exists():
    print(f"{QC_SAMPLE_PATH.name} exists; sample not redrawn")
else:
    articles = pd.read_csv(NEWS_DIR / "articles.csv", dtype=str, keep_default_na=False)
    write_qc_sample(draw_qc_sample(articles, per_source=PER_SOURCE, seed=SEED), QC_SAMPLE_PATH)
    print(f"articles in pool: {len(articles)}")

sample = pd.read_csv(QC_SAMPLE_PATH, dtype=str, keep_default_na=False)
print(f"sampled: {len(sample)}")
print(sample["source"].value_counts().sort_index().to_string())

## Step 1b: Add the market hand-label columns

Adds `my_relevance_market` and `my_sentiment_market` (empty) to an existing `qc_sample.csv`. The same 25 rows and every existing value are kept; a file that already has the columns is not rewritten.

In [ ]:
added = add_missing_label_columns(QC_SAMPLE_PATH)
print(f"columns added: {added}")
print(list(pd.read_csv(QC_SAMPLE_PATH, dtype=str, keep_default_na=False).columns))

## Stage 2: comparison with hand labels

Fields compared against the hand labels (model run 1):
- relevance agreement for market, BTC and SOL;
- on articles both marked relevant: sentiment direction agreement (negative / neutral / positive) and share within one step, for market and BTC;
- `event_type` agreement;
- the same figures for CoinDesk (title only) and for all other sources;
- run-1 versus run-2 identical share for every label field;
- every disagreement, with the article title, summary, hand label and model label.

An empty hand sentiment cell is null. Shares come with standard errors sqrt(p(1-p)/n); `chance` is the agreement expected from the two marginal distributions.

## Step 2: Label the sample twice

Pinned model `gpt-oss-120b` (the API must report `openai/gpt-oss-120b`; any other reported model is not stored and stops labeling) and the current prompt, temperature 0. Writes only to `qc_labels.csv` (never `labels.csv`). Requests are spaced 10 s apart for this QC run (the production default in `label.py` is unchanged). Rows already present for the same run, article, model and prompt version are skipped, so re-running this cell does not label again; if the API fails, labeling stops and the next run resumes.

`qc_labels.csv` is append-only and still holds one earlier `gemini-3.8-flash` row; all comparisons below use only rows from the pinned model and current prompt version.

In [ ]:
from cryptosense.news.label import MODEL, PROMPT_VERSION, read_env_key
from cryptosense.news.qc import (
    agreement, current_model_labels, disagreements, label_qc_sample, merge_hand_and_model,
    run_consistency,
)

QC_LABELS_PATH = NEWS_DIR / "qc_labels.csv"
QC_BACKOFF_S = 10  # seconds between requests, this QC run only
result = label_qc_sample(read_env_key(), sample, QC_LABELS_PATH, runs=2, backoff_s=QC_BACKOFF_S)
print(f"model {MODEL}, prompt_version {PROMPT_VERSION}")
print(result)

labels = pd.read_csv(QC_LABELS_PATH, dtype=str, keep_default_na=False)
print("all rows by model_requested:", labels["model_requested"].value_counts().to_dict())
current = labels[(labels["model_requested"] == MODEL) & (labels["prompt_version"] == PROMPT_VERSION)]
print(current.groupby("run").agg(rows=("article_id", "size"),
                                 invalid=("invalid_reason", lambda s: (s != "").sum())).to_string())
print("model_reported (current model):", current["model_reported"].value_counts().to_dict())
print("valid rows compared:", len(current_model_labels(labels)))

## Step 3: Agreement with hand labels (run 1)

In [ ]:
merged = merge_hand_and_model(sample, labels, run=1)
pd.set_option("display.width", 200)
print(f"articles compared: {len(merged)}\n")
print("All sources")
print(agreement(merged).round(3).to_string())
is_coindesk = merged["source"] == "coindesk"
print("\nCoinDesk (title only)")
print(agreement(merged[is_coindesk]).round(3).to_string())
print("\nAll other sources")
print(agreement(merged[~is_coindesk]).round(3).to_string())

## Step 4: Run 1 versus run 2

In [ ]:
print(run_consistency(labels, 1, 2).round(3).to_string())

## Step 5: Every disagreement (hand versus run 1)

In [ ]:
found = disagreements(merged)
print(f"disagreements: {len(found)}\n")
for i, row in enumerate(found.itertuples(index=False), 1):
    print(f"{i}. [{row.field}] {row.source}: {row.title}")
    print(f"   summary: {row.summary}")
    print(f"   hand: {row.hand}   model: {row.model}\n")